# NOTEBOOK 01: DATA MERGING & FEATURE ENGINEERING (OULAD & KAGGLE S2)

**Dự án:** Hệ thống Cảnh báo sớm Học vụ và Mô phỏng Kịch bản Học tập 2 Tầng (What-If Simulator)

**Mục tiêu Notebook 1:**
- Gộp 4 bảng cốt lõi của bộ dữ liệu OULAD (`studentInfo`, `assessments`, `studentAssessment`, `studentVle`).
- Áp dụng quy chuẩn đào tạo tín chỉ Việt Nam (tỷ lệ điểm Quá trình - Cuối kỳ 40-60 / 50-50, thang điểm 10).
- Xây dựng 3 đặc trưng học thuật chuyên sâu: `weighted_final_score`, `gpa_credits`, `gap_process_exam`.
- Xử lý bộ dữ liệu Tầng 2 (`StudentPerformanceFactors.csv`) và đồng bộ thang điểm 10.
- **Nguyên tắc kỹ thuật sống còn:** Tối ưu bộ nhớ RAM (Groupby trước -> Merge sau), chống rò rỉ dữ liệu (Anti-Data Leakage bằng cách loại bỏ bài thi Exam khỏi điểm quá trình).

---
## 📦 EPIC 1: DATA ENGINEERING & FEATURE ENGINEERING
### 🔹 Task 1.1: Kiểm tra toàn vẹn khóa chính & Làm sạch bảng OULAD gốc

**Mục tiêu:**
1. Xác thực bộ 3 khóa định danh composite `KEYS = ['id_student', 'code_module', 'code_presentation']` trên `studentInfo.csv` không bị trùng lặp.
2. Xử lý các dòng điểm bị thiếu (`NaN`) trong `studentAssessment.csv` bằng cách điền `0.0` (phản ánh sinh viên bỏ không nộp bài).
3. **Chống Data Leakage:** Loại trừ toàn bộ các bài đánh giá có `assessment_type == 'Exam'` khỏi điểm quá trình, vì sinh viên bỏ học (`Withdrawn`) sẽ không có điểm thi cuối kỳ. Nếu giữ lại điểm Exam, mô hình sẽ bị rò rỉ nhãn.
4. Ghép (`merge`) bảng điểm với `assessments.csv` để lấy thông tin môn học, hạn nộp (`date`) và trọng số (`weight`).
5. Quy đổi toàn bộ điểm về **thang điểm 10** (`score / 10.0`).
6. Chia quá trình học tập thành 2 giai đoạn: **Nửa đầu kỳ (`early`: date <= 100)** và **Nửa cuối kỳ (`late`: date > 100)** để chuẩn bị cho việc tính độ lệch phong độ.

In [ ]:
# CELL 1: Import thư viện và thiết lập cấu hình
import os
import gc
import pandas as pd
import numpy as np

# Bộ 3 khóa liên kết chuẩn trên OULAD
KEYS = ['id_student', 'code_module', 'code_presentation']
RAW_S1 = os.path.join('data', 'raw', 'S1')
RAW_S2 = os.path.join('data', 'raw', 'S2')

print("Đã import thư viện")


### 1.1.1. Kiểm tra toàn vẹn khóa chính trên `studentInfo.csv`

Trong OULAD, một sinh viên (`id_student`) có thể đăng ký nhiều môn (`code_module`) và qua nhiều học kỳ (`code_presentation`). Do đó, bản ghi chỉ duy nhất khi xét trên cả bộ 3 khóa này.

In [ ]:
# CELL 2: Nạp studentInfo và kiểm tra trùng lặp khóa chính
df_info = pd.read_csv(os.path.join(RAW_S1, 'studentInfo.csv'))

# Kiểm tra số lượng bản ghi trùng khóa
dup_count = df_info.duplicated(subset=KEYS).sum()
assert dup_count == 0, f"❌ Lỗi: Có {dup_count} dòng trùng lặp khóa chính!"

print(f"✅ Bảng studentInfo.csv: {len(df_info):,} dòng.")
print(f"✅ Toàn vẹn khóa chính KEYS={KEYS}: {dup_count} trùng lặp (100% Unique).")
print(f"   Số lượng sinh viên duy nhất: {df_info['id_student'].nunique():,}")
print(f"   Số lượng môn học: {df_info['code_module'].nunique()} môn ({list(df_info['code_module'].unique())})")

### 1.1.2. Làm sạch bảng điểm & Chống Rò rỉ Dữ liệu (Anti-Data Leakage)

> ⚠️ **Cảnh báo Data Leakage:** Bảng `assessments.csv` chứa 3 loại bài kiểm tra:
> - `TMA` (Tutor Marked Assessment): Bài tập quá trình do giảng viên chấm.
> - `CMA` (Computer Marked Assessment): Bài kiểm tra trắc nghiệm máy tính chấm.
> - `Exam`: Bài thi kết thúc học phần.
>
> Trong thực tế OULAD, sinh viên bỏ học (`Withdrawn`) hầu hết rút lui trước hoặc trong kỳ thi, nên không có điểm thi `Exam`. Nếu đưa bài thi `Exam` vào tính điểm quá trình để dự báo bỏ học, mô hình sẽ bị rò rỉ dữ liệu (chỉ cần nhìn thấy thiếu điểm Exam là đoán ngay bỏ học). Vì vậy, ta **lọc bỏ hoàn toàn các bài Exam** và chỉ giữ lại bài quá trình (`TMA` và `CMA`).

In [ ]:
# CELL 3: Đọc bảng điểm, loại bỏ Exam và quy đổi về thang điểm 10
df_assess = pd.read_csv(os.path.join(RAW_S1, 'assessments.csv'))
df_sa = pd.read_csv(os.path.join(RAW_S1, 'studentAssessment.csv'))

print(f"Tổng số bài kiểm tra cấu hình trong assessments.csv: {len(df_assess):,}")
print(f"Phân bố loại bài:\n{df_assess['assessment_type'].value_counts()}\n")

# Xử lý missing values trong điểm số: sinh viên có tên trong danh sách nhưng không nộp bài -> điền 0.0
nan_scores = df_sa['score'].isna().sum()
df_sa['score'] = df_sa['score'].fillna(0.0)
print(f"✅ Đã xử lý {nan_scores} dòng điểm khuyết thiếu (NaN) -> điền 0.0")

# [ANTI-DATA LEAKAGE] Loại bỏ toàn bộ bài kiểm tra loại 'Exam'
df_assess_process = df_assess[df_assess['assessment_type'].isin(['TMA', 'CMA'])].copy()
num_exam_removed = (df_assess['assessment_type'] == 'Exam').sum()
print(f"🛡️ [ANTI-DATA LEAKAGE] Đã loại bỏ {num_exam_removed} bài Exam khỏi dữ liệu quá trình.")
print(f"   Chỉ giữ lại {len(df_assess_process)} bài đánh giá quá trình (TMA & CMA).")

# Merge lấy thông tin môn học, kỳ học, hạn nộp (date) và trọng số (weight)
df_sa_full = pd.merge(
    df_sa,
    df_assess_process[['id_assessment', 'code_module', 'code_presentation', 'assessment_type', 'date', 'weight']],
    on='id_assessment',
    how='inner'  # Inner join tự động loại trừ các bài thi Exam
)

# Quy đổi điểm từ thang 100 về thang điểm 10 chuẩn Việt Nam
df_sa_full['score_10'] = (df_sa_full['score'] / 10.0).round(2)

# Chia 2 giai đoạn học kỳ: Early (date <= 100) vs Late (date > 100)
# Ngày 100 tương ứng khoảng kết thúc nửa đầu học kỳ (học kỳ OULAD kéo dài ~240-269 ngày)
df_sa_full['phase'] = np.where(df_sa_full['date'] <= 100, 'early', 'late')

# ============================================================
# CHECKPOINTS KIỂM ĐỊNH CHẤT LƯỢNG (ACCEPTANCE CRITERIA)
# ============================================================
assert (df_sa_full['assessment_type'] == 'Exam').sum() == 0, "❌ Lỗi: Còn sót bài thi Exam!"
assert df_sa_full['score_10'].between(0.0, 10.0).all(), "❌ Lỗi: Điểm số vượt ngoài khoảng [0, 10]!"

print("\n" + "="*60)
print("🎉 TASK 1.1 CHECKPOINTS: TẤT CẢ TIÊU CHÍ ĐỀU ĐẠT CHUẨN!")
print(f"   Tổng số lượt nộp bài quá trình: {len(df_sa_full):,} dòng")
print(f"   Thang điểm score_10: min={df_sa_full['score_10'].min()}, max={df_sa_full['score_10'].max()}, mean={df_sa_full['score_10'].mean():.2f}")
print(f"   Phân bố giai đoạn nộp bài:\n{df_sa_full['phase'].value_counts()}")
print("="*60)

---
### 🔹 Task 1.2: Gom nhóm (Groupby) hành vi tương tác VLE chống tràn RAM

**Bối cảnh kỹ thuật:**
- Bảng `studentVle.csv` chứa hơn **10.6 triệu dòng** ghi lại từng lượt click chuột của sinh viên.
- Nếu dùng `pd.read_csv` mặc định không ép kiểu và merge trực tiếp vào `studentInfo`, bộ nhớ sẽ ngốn >2GB RAM và làm sập kernel.

**Kỹ thuật chuẩn Data Engineer:**
1. **Chỉ nạp các cột cần thiết (`usecols`):** Bỏ qua `id_site` để tiết kiệm RAM.
2. **Ép kiểu dữ liệu nhẹ (`dtypes`):** Chuyển `code_module`, `code_presentation` sang `category`, `id_student` và `sum_click` sang `int32`, `date` sang `int16`. Việc này giảm dung lượng từ >1.2GB xuống còn **~128MB** (giảm gần 90% bộ nhớ).
3. **Gom nhóm trước khi gộp (Groupby before Merge):** Rút gọn từ 10.6 triệu dòng về đúng **29,228 bản ghi** sinh viên duy nhất.
4. **Giải phóng RAM triệt để (`gc.collect()`):** Xóa ngay dataframe thô `df_vle` sau khi tổng hợp.
5. **Tạo 3 đặc trưng tương tác học tập:**
   - `attendance_rate`: Quy đổi `active_days` sang tỷ lệ chuyên cần % dựa trên phân vị thứ 95 (P95) của từng môn học (`code_module`), trần 100%.
   - `vle_engagement_score`: Điểm số tương tác học liệu LMS trên thang điểm 0 - 10 (áp dụng log transform để giảm ảnh hưởng của ngoại lai click nhiều).
   - `vle_drop_ratio`: Tỷ lệ sụt giảm tương tác giai đoạn nửa sau học kỳ so với nửa đầu kỳ trong khoảng `[-1.0, 1.0]` (chỉ số cảnh báo nguy cơ bỏ học).

CELL 4: Đọc studentVle.csv với usecols và dtypes tối ưu RAM

In [ ]:
# CELL 4: Đọc studentVle.csv với usecols và dtypes tối ưu RAM
import time

vle_dtypes = {
    'code_module': 'category',
    'code_presentation': 'category',
    'id_student': 'int32',
    'date': 'int16',
    'sum_click': 'int32'
}

t_start = time.time()
print("⏳ Đang nạp studentVle.csv với định dạng tối ưu RAM...")

df_vle = pd.read_csv(
    os.path.join(RAW_S1, 'studentVle.csv'),
    usecols=['code_module', 'code_presentation', 'id_student', 'date', 'sum_click'],
    dtype=vle_dtypes
)

ram_usage_mb = df_vle.memory_usage(deep=True).sum() / (1024 * 1024)
print(f"✅ Đã nạp thành công {len(df_vle):,} dòng chỉ trong {time.time() - t_start:.2f}s!")
print(f"   Bộ nhớ RAM tiêu tốn: chỉ {ram_usage_mb:.1f} MB (thay vì >1.2 GB nếu để kiểu mặc định).")

CELL 5: Tách 2 giai đoạn (early/late), GroupBy và giải phóng RAM

In [ ]:
# CELL 5: Tách 2 giai đoạn (early/late), GroupBy và giải phóng RAM
# Phân chia tương tác: early (date <= 100) và late (date > 100)
df_vle['phase'] = np.where(df_vle['date'] <= 100, 'early', 'late')

t_groupby = time.time()
print("⏳ Đang thực hiện GroupBy tổng hợp số click và số ngày học...")

# 1. Tổng số click toàn kỳ và số ngày học thực tế
agg_vle = df_vle.groupby(KEYS, observed=True).agg(
    total_clicks=('sum_click', 'sum'),
    active_days=('date', 'nunique')
).reset_index()

# 2. Tổng số click ở từng giai đoạn: clicks_early và clicks_late
clicks_phase = df_vle.groupby(KEYS + ['phase'], observed=True)['sum_click'].sum().unstack(fill_value=0)
if 'early' in clicks_phase.columns and 'late' in clicks_phase.columns:
    clicks_phase = clicks_phase[['early', 'late']].rename(columns={'early': 'clicks_early', 'late': 'clicks_late'})
elif 'early' in clicks_phase.columns:
    clicks_phase = clicks_phase[['early']].rename(columns={'early': 'clicks_early'})
    clicks_phase['clicks_late'] = 0
else:
    clicks_phase['clicks_early'] = 0
    clicks_phase['clicks_late'] = clicks_phase['late']

agg_vle = agg_vle.merge(clicks_phase.reset_index(), on=KEYS, how='left')
agg_vle['clicks_early'] = agg_vle['clicks_early'].fillna(0).astype('int32')
agg_vle['clicks_late'] = agg_vle['clicks_late'].fillna(0).astype('int32')

# 3. GIẢI PHÓNG BỘ NHỚ RAM NGAY LẬP TỨC (Data Engineer Rule)
del df_vle
del clicks_phase
gc.collect()

print(f"✅ GroupBy hoàn tất trong {time.time() - t_groupby:.2f}s!")
print(f"   Rút gọn từ 10,655,280 dòng -> {len(agg_vle):,} bản ghi sinh viên duy nhất.")
print("   🧹 Đã dọn dẹp và giải phóng hoàn toàn bộ nhớ bảng raw studentVle.")

CELL 6: Tính attendance_rate (P95 môn), vle_engagement_score và vle_drop_ratio


In [ ]:
# CELL 6: Tính attendance_rate (P95 môn), vle_engagement_score và vle_drop_ratio
# 1. attendance_rate (%): Chuẩn hóa theo phân vị thứ 95 (P95) của từng môn học (code_module)
p95_per_module = agg_vle.groupby('code_module', observed=True)['active_days'].quantile(0.95).rename('p95_active_days')
agg_vle = agg_vle.merge(p95_per_module, on='code_module', how='left')

agg_vle['attendance_rate'] = (
    (agg_vle['active_days'] / agg_vle['p95_active_days']) * 100.0
).clip(upper=100.0).round(1)

# 2. vle_engagement_score: Thang 0 - 10 dựa trên log-transform của total_clicks
agg_vle['log_clicks'] = np.log1p(agg_vle['total_clicks'])
p95_log_clicks = agg_vle['log_clicks'].quantile(0.95)
agg_vle['vle_engagement_score'] = (
    (agg_vle['log_clicks'] / p95_log_clicks) * 10.0
).clip(upper=10.0).round(2)

# 3. vle_drop_ratio: Tỷ lệ sụt giảm tương tác giai đoạn nửa sau học kỳ (giới hạn trong [-1.0, 1.0])
# - Giá trị dương (+) -> Tương tác giảm mạnh ở cuối kỳ (Dấu hiệu nguy cơ bỏ học)
# - Giá trị âm (-)   -> Tương tác tăng vọt ở cuối kỳ (Học nước rút / ôn thi tích cực)
agg_vle['vle_drop_ratio'] = (
    (agg_vle['clicks_early'] - agg_vle['clicks_late']) / (agg_vle['clicks_early'] + 1.0)
).clip(lower=-1.0, upper=1.0).round(4)

# CHECKPOINTS KIỂM ĐỊNH CHẤT LƯỢNG (ACCEPTANCE CRITERIA)
assert len(agg_vle) <= len(df_info), "❌ Lỗi: Số dòng agg_vle lớn hơn số sinh viên trong studentInfo!"
assert agg_vle['attendance_rate'].between(0.0, 100.0).all(), "❌ Lỗi: attendance_rate vượt ngoài [0%, 100%]!"
assert agg_vle['vle_engagement_score'].between(0.0, 10.0).all(), "❌ Lỗi: vle_engagement_score vượt ngoài [0, 10]!"
assert agg_vle['vle_drop_ratio'].between(-1.0, 1.0).all(), "❌ Lỗi: vle_drop_ratio vượt ngoài [-1.0, 1.0]!"

print("\n" + "="*60)
print(f"   Số bản ghi sau gom nhóm: {len(agg_vle):,} sinh viên")
print(f"   Chuyên cần attendance_rate: min={agg_vle['attendance_rate'].min()}%, max={agg_vle['attendance_rate'].max()}%, mean={agg_vle['attendance_rate'].mean():.1f}%")
print(f"   Điểm tương tác vle_engagement_score: min={agg_vle['vle_engagement_score'].min()}, max={agg_vle['vle_engagement_score'].max()}, mean={agg_vle['vle_engagement_score'].mean():.2f}/10")
print(f"   Tỷ lệ sụt giảm vle_drop_ratio: min={agg_vle['vle_drop_ratio'].min()}, max={agg_vle['vle_drop_ratio'].max()}, mean={agg_vle['vle_drop_ratio'].mean():.3f}")
print("="*60)

---
### 🔹 Task 1.3: Thiết lập 3 đặc trưng Tín chỉ Việt Nam & Gộp bảng Master Tầng 1

**Mục tiêu học thuật cốt lõi:**
1. **Tính điểm quá trình 2 giai đoạn (`score_early_sem` & `score_late_sem`):** Dùng kỹ thuật Vectorized GroupBy để tính điểm trung bình có trọng số các bài TMA/CMA ở nửa đầu kỳ (`date <= 100`) và nửa cuối kỳ (`date > 100`).
2. **Ghép bảng `studentRegistration.csv`:** Bổ sung `date_registration` và `date_unregistration`.  
   > 🛡️ **Nguyên tắc chống Target Leakage:** Biến `date_unregistration` có tương quan 99.1% với việc bỏ học (chỉ xuất hiện sau khi sinh viên đã nộp đơn thôi học). Ta giữ cột này trên bảng Master để phục vụ thống kê EDA, nhưng **tuyệt đối không đưa vào `FEATURES_T1` huấn luyện mô hình**.
3. **Tạo 3 đặc trưng học vụ Tín chỉ Việt Nam:**
   - `weighted_final_score`: Điểm tổng kết trọng số (tỷ lệ 50-50 cho khối STEM: `CCC`, `DDD`, `EEE`, `FFF`; tỷ lệ 40-60 cho khối Đại cương: `AAA`, `BBB`, `GGG`).
   - `gpa_credits`: Điểm tích lũy theo tải trọng tín chỉ `studied_credits` và số lần học lại `num_of_prev_attempts`.
   - `credit_burden_risk`: Chỉ số áp lực nợ môn khi điểm quá trình dưới 5.0.
   - `gap_process_exam`: Độ lệch điểm đầu kỳ - cuối kỳ (`score_early_sem - score_late_sem`).
4. **Gộp bảng Master hoàn chỉnh (`LEFT JOIN`):** Lấy `studentInfo.csv` làm gốc (32,593 dòng), xử lý triệt để missing values.
5. **Tạo nhãn mục tiêu `is_dropout`:** `1` (Withdrawn, Fail) vs `0` (Pass, Distinction).
6. **Lưu file Master:** Xuất ra `data/processed/stage1_oulad_master.csv`.

In [ ]:
# CELL 7: Tính điểm quá trình theo giai đoạn (Vectorized GroupBy)
# Tính điểm có trọng số từng phần: score_10 * weight
df_sa_full['weighted_score_part'] = df_sa_full['score_10'] * df_sa_full['weight']

# Gom nhóm tính tổng trọng số và tổng điểm có trọng số theo từng sinh viên & giai đoạn
phase_agg = df_sa_full.groupby(KEYS + ['phase'], observed=True).agg(
    sum_weighted=('weighted_score_part', 'sum'),
    sum_weight=('weight', 'sum'),
    mean_score=('score_10', 'mean'),
    count=('id_assessment', 'count')
).reset_index()

# Tính điểm trung bình có trọng số cho từng giai đoạn
phase_agg['phase_score'] = np.where(
    phase_agg['sum_weight'] > 0,
    phase_agg['sum_weighted'] / phase_agg['sum_weight'],
    phase_agg['mean_score']
).round(2)

# Pivot bảng để có 2 cột riêng biệt: score_early_sem và score_late_sem
score_pivot = phase_agg.pivot(index=KEYS, columns='phase', values='phase_score').reset_index()
score_pivot.columns.name = None
if 'early' not in score_pivot.columns: score_pivot['early'] = np.nan
if 'late' not in score_pivot.columns:  score_pivot['late'] = np.nan
score_pivot = score_pivot.rename(columns={'early': 'score_early_sem', 'late': 'score_late_sem'})

# Tính tổng số bài kiểm tra quá trình đã nộp toàn kỳ
submitted_total = phase_agg.groupby(KEYS)['count'].sum().reset_index(name='submitted_count')
score_final = score_pivot.merge(submitted_total, on=KEYS, how='left')

print(f"✅ Tính toán điểm quá trình hoàn tất! Bảng điểm tổng hợp: {len(score_final):,} sinh viên có điểm.")
print(f"   Điểm đầu kỳ trung bình: {score_final['score_early_sem'].mean():.2f}/10")
print(f"   Điểm cuối kỳ trung bình: {score_final['score_late_sem'].mean():.2f}/10")

In [ ]:
# CELL 8: Ghép nối Master Tầng 1 (studentInfo + studentRegistration + Điểm + VLE)
# 1. Đọc bảng studentRegistration
df_reg = pd.read_csv(os.path.join(RAW_S1, 'studentRegistration.csv'))

# 2. Ghép nối tuần tự bằng LEFT JOIN với df_info làm gốc (32,593 dòng)
df_master = df_info.merge(df_reg[KEYS + ['date_registration', 'date_unregistration']], on=KEYS, how='left')
df_master = df_master.merge(score_final, on=KEYS, how='left')
df_master = df_master.merge(
    agg_vle[KEYS + ['total_clicks', 'active_days', 'attendance_rate', 'vle_engagement_score', 'vle_drop_ratio', 'clicks_early', 'clicks_late']],
    on=KEYS, how='left'
)

# 3. Xử lý Missing Values theo đúng bản chất học vụ:
# - Sinh viên bỏ học sớm không nộp bài / không click VLE -> điền 0
FILL_ZERO_COLS = [
    'score_early_sem', 'score_late_sem', 'submitted_count',
    'total_clicks', 'active_days', 'attendance_rate',
    'vle_engagement_score', 'vle_drop_ratio', 'clicks_early', 'clicks_late'
]
df_master[FILL_ZERO_COLS] = df_master[FILL_ZERO_COLS].fillna(0)

# - date_registration: điền giá trị trung vị (median) theo từng học kỳ
df_master['date_registration'] = df_master.groupby('code_presentation')['date_registration'].transform(lambda s: s.fillna(s.median()))

# - imd_band: điền 'Unknown' cho 1,111 dòng khuyết thiếu
df_master['imd_band'] = df_master['imd_band'].fillna('Unknown')

print(f"✅ Ghép bảng Master thành công: {df_master.shape[0]:,} dòng x {df_master.shape[1]} cột.")

In [ ]:
# CELL 9: Xây dựng 3 đặc trưng Tín chỉ Việt Nam & Tạo nhãn mục tiêu is_dropout
# -------------------------------------------------------------------------
# ĐẶC TRƯNG 1: weighted_final_score (Điểm tổng kết trọng số chuẩn 4-6 / 5-5)
# - Môn chuyên ngành/STEM (CCC, DDD, EEE, FFF): tỷ lệ 50% đầu kỳ + 50% cuối kỳ
# - Môn đại cương/xã hội (AAA, BBB, GGG): tỷ lệ 40% đầu kỳ + 60% cuối kỳ
# -------------------------------------------------------------------------
STEM_MODULES = ['CCC', 'DDD', 'EEE', 'FFF']
df_master['weighted_final_score'] = np.where(
    df_master['code_module'].isin(STEM_MODULES),
    0.5 * df_master['score_early_sem'] + 0.5 * df_master['score_late_sem'],
    0.4 * df_master['score_early_sem'] + 0.6 * df_master['score_late_sem']
).round(2)

# -------------------------------------------------------------------------
# ĐẶC TRƯNG 2: gpa_credits (Điểm tích lũy điều chỉnh theo tải trọng tín chỉ)
# Phản ánh áp lực: học nhiều tín chỉ (studied_credits) + từng nợ môn (num_of_prev_attempts)
# -------------------------------------------------------------------------
df_master['gpa_credits'] = (
    df_master['weighted_final_score']
    * (df_master['studied_credits'] / 60.0)
    / (1.0 + 0.5 * df_master['num_of_prev_attempts'])
).round(4)

# Biến phụ trợ: credit_burden_risk (Áp lực nợ môn khi điểm < 5.0)
df_master['credit_burden_risk'] = (
    df_master['studied_credits']
    * (1 + df_master['num_of_prev_attempts'])
    * np.maximum(0.0, 5.0 - df_master['weighted_final_score'])
).round(4)

# -------------------------------------------------------------------------
# ĐẶC TRƯNG 3: gap_process_exam (Độ lệch phong độ đầu kỳ - cuối kỳ)
# - Dương (+): Phong độ tụt dốc (đầu kỳ cao, cuối kỳ bỏ bê/rớt điểm)
# - Âm (-): Phong độ cải thiện (cuối kỳ học tập bứt phá)
# -------------------------------------------------------------------------
df_master['gap_process_exam'] = (
    df_master['score_early_sem'] - df_master['score_late_sem']
).round(4)

# -------------------------------------------------------------------------
# NHÃN MỤC TIÊU: is_dropout
# - 1 (Nguy cơ học vụ): final_result thuộc ['Withdrawn', 'Fail']
# - 0 (An toàn):        final_result thuộc ['Pass', 'Distinction']
# -------------------------------------------------------------------------
df_master['is_dropout'] = df_master['final_result'].isin(['Withdrawn', 'Fail']).astype(int)

# Xuất file Master ra thư mục processed
os.makedirs(os.path.join('data', 'processed'), exist_ok=True)
master_out_path = os.path.join('data', 'processed', 'stage1_oulad_master.csv')
df_master.to_csv(master_out_path, index=False)
print(f"💾 Đã lưu thành công Master Tầng 1: {master_out_path}")

# ============================================================
# CHECKPOINTS KIỂM ĐỊNH CHẤT LƯỢNG (ACCEPTANCE CRITERIA)
# ============================================================
assert len(df_master) == len(df_info), f"❌ Lỗi: Số dòng không khớp studentInfo ({len(df_master)} vs {len(df_info)})!"
assert df_master['weighted_final_score'].between(0.0, 10.0).all(), "❌ Lỗi: weighted_final_score ngoài [0, 10]!"

FEATURES_T1 = [
    'weighted_final_score', 'gpa_credits', 'gap_process_exam',
    'submitted_count', 'attendance_rate', 'total_clicks',
    'num_of_prev_attempts', 'studied_credits',
    'vle_engagement_score', 'vle_drop_ratio'
]
assert df_master[FEATURES_T1].isna().sum().sum() == 0, "❌ Lỗi: Có giá trị null trong FEATURES_T1!"

print("\n" + "="*60)
print("🎉 TASK 1.3 CHECKPOINTS: TẤT CẢ TIÊU CHÍ ĐỀU ĐẠT CHUẨN XUẤT SẮC!")
print(f"   Tổng số dòng master: {len(df_master):,} (khớp 100% với studentInfo)")
print(f"   Phân bố nhãn mục tiêu is_dropout:\n{df_master['is_dropout'].value_counts()}\n")
print(f"   Tỷ lệ nguy cơ bỏ học: {df_master['is_dropout'].mean()*100:.1f}%")
print("   Thống kê 10 đặc trưng chính FEATURES_T1:")
print(df_master[FEATURES_T1].describe().T[['min', 'mean', 'max']])
print("="*60)